In [1]:
import sys

sys.path.append('..')

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from pathlib import Path
from src.features.transformers import CutOrdinalEncoder

# 1. Загрузка данных

In [2]:
df = pd.read_csv('../data/raw/diamonds.csv')
if 'Unnamed: 0' in df.columns:
  df = df.drop(columns=['Unnamed: 0'])

X = df.drop(columns=['price'])
y = df['price']

# 2. Train / Test split (80/20)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Определение колонок

In [4]:
# Признак cut после CutOrdinalEncoder станет числовым
num_cols = ['carat', 'depth', 'table', 'x', 'y', 'z', 'cut']
cat_cols = ['color', 'clarity']

# 4. Предобработка

In [5]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols),
    ]
)

# 5. Итоговый пайплайн

In [6]:
full_pipeline = Pipeline([
    ('cut_encoder', CutOrdinalEncoder(column_name='cut')),
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42)),
])


# 6. Обучение

In [7]:
# Задаем сетку параметров для шага 'regressor'
# Синтаксис: <имя_шага_в_пайплайне>__<имя_параметра>
param_grid = {
    'regressor__n_estimators': [50, 100],
    'regressor__max_depth': [15, 25],
    'regressor__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=3,                                     # 3 фолда кросс-валидации
    scoring='neg_root_mean_squared_error',    # оптимизируем RMSE
    n_jobs=-1,                                # задействуем все ядра процессора
    verbose=2
)

# Обучение сетки
grid_search.fit(X_train, y_train)

print(f"\nЛучшие параметры: {grid_search.best_params_}")
print(f"Лучший RMSE на кросс-валидации: {-grid_search.best_score_:.2f}")

# Извлекаем лучший обученный пайплайн
best_pipeline = grid_search.best_estimator_

Fitting 3 folds for each of 8 candidates, totalling 24 fits

Лучшие параметры: {'regressor__max_depth': 25, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}
Лучший RMSE на кросс-валидации: 563.34


# 7. Оценка качества

In [8]:
y_pred = best_pipeline.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'RMSE: {rmse:.2f}')
print(f'MAE:  {mae:.2f}')
print(f'R^2:  {r2:.4f}')

RMSE: 546.53
MAE:  268.67
R^2:  0.9812


# 8. Сохранение модели

In [9]:
joblib.dump(best_pipeline, '../models/pipeline.joblib', compress=3)

['../models/pipeline.joblib']